## Generating Ground Truth Data

### Loading the documents

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm =[]

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


### Generating questions with structured output

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instruction = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json
user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content":data_gen_instruction},
    {"role": "user", "content":user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed
print(result)

questions=['I just found this course late — can I still enroll and follow along?', 'Is it too late to join the class if I discovered it after it started?', 'If I start the course now, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I join late?', 'Are project submissions still open for people who are joining the course now?']


In [12]:
print(result.questions)

['I just found this course late — can I still enroll and follow along?', 'Is it too late to join the class if I discovered it after it started?', 'If I start the course now, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I join late?', 'Are project submissions still open for people who are joining the course now?']


### Reusable utilities

In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instruction,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still possible to join now, or did I miss the start?', 'Can I enroll late in the course and still follow along with the materials?', 'If I start the course after it already began, am I still eligible for a certificate?', 'What’s the deadline for project submission if I want to get the certificate?', 'Is it okay to join the course now, and what do I need to do to qualify for a certificate?']


In [15]:
usage.input_tokens, usage.output_tokens

(207, 107)

In [16]:
from evaluation_utils import calc_price

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.0004815, 'total_cost': 0.00063675}

In [18]:
records =[]

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

In [19]:
records

[{'question': 'I just found this course — is it still possible to join now, or did I miss the start?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll late in the course and still follow along with the materials?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course after it already began, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course now, and what do I need to do to qualify for a certificate?',
  'document': '74eb249bbf'}]

## Generating Ground Truth for All Documents

In [20]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instruction,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })
    
    return results, usage

In [22]:
from tqdm.auto import tqdm

In [23]:
ground_truth = []
usages =[]

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

### Parallel Processing

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [26]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.087486

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.087486

In [29]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("/workspaces/intro-to-RAG/04-evaluation/data/ground_truth-new.csv", index=False)

## Search Evaluation

In [31]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [32]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [33]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [34]:
q = ground_truth[0]
q

{'question': 'I just found this course — is it still okay to join now?',
 'document': '74eb249bbf'}

In [35]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [36]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [37]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [38]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [39]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — is it still okay to join now?


[1, 0, 0, 0, 0]

In [40]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the live video URL for workshop or Office Hours sessions before they start?


[1, 0, 0, 0, 0]

In [41]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [42]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [43]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [44]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [45]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [47]:
relevance_total = compute_relevance_total(ground_truth, text_search)


  0%|          | 0/565 [00:00<?, ?it/s]

### Search Evaluation Metrics

#### Hit Rate

In [49]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1
cnt

481

In [50]:
cnt/len(relevance_total)

0.8513274336283185

In [51]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    
    return cnt/len(relevance)

In [52]:
hit_rate(relevance_total)

0.8513274336283185

#### Mean Reciprocal Rank (MRR)

In [53]:
total_score = 0.0

for line in relevance_total:
    for rank in range(len(line)): 
        if line[rank] == 1:
            total_score = total_score + 1/(rank+1)
            break
total_score

408.16666666666646

In [54]:
total_score/len(relevance_total)

0.7224188790560468

In [55]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)): 
            if line[rank] == 1:
                total_score = total_score + 1/(rank+1)
                break
    
    return total_score/len(relevance)

In [56]:
mrr(relevance_total)

0.7224188790560468

#### Reusable Evaluation Function

In [59]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [60]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8513274336283185, 'mrr': 0.7224188790560468}

### Search Parameter Tuning


In [61]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [62]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8955752212389381, 'mrr': 0.794955752212389}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9008849557522124, 'mrr': 0.7948672566371678}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8513274336283185, 'mrr': 0.7224188790560468}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8212389380530973, 'mrr': 0.6834218289085543}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.784070796460177, 'mrr': 0.6537758112094393}


In [63]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [64]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [65]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.971681,0.878732
18,2.0,4.0,0.1,0.971681,0.878732
3,1.0,2.0,0.1,0.971681,0.878024
35,5.0,10.0,0.5,0.971681,0.878024
19,2.0,4.0,0.2,0.971681,0.878024
4,1.0,2.0,0.2,0.971681,0.877139
33,5.0,10.0,0.1,0.969912,0.875723
20,2.0,4.0,0.5,0.969912,0.872950
8,1.0,4.0,0.5,0.973451,0.867463
6,1.0,4.0,0.1,0.964602,0.866667


In [66]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )